In [1]:
from pathlib import Path
import h5py
from dataset_enhancement import save_h5
import stemdiff as sd

result_dir = Path("dataset_filtered")
result_dir.mkdir(exist_ok=True)

# Filtering

In [2]:
paths = {
    "au": Path("../DATA.STEMDIFF/1_AU/EX1.AU/DATA"),
    "tbf3": Path("../DATA.STEMDIFF/2_TBF3/VZ2.TBF3.R2"),
    "feo": Path("../DATA.STEMDIFF/3_FEO_PURE/FeO-Pure_Cimc"),
    "laf3": Path("../DATA.STEMDIFF/4_MARUSKA_LAF3/D_MARUSKA_C214"),
    "gdf3": Path("../DATA.STEMDIFF/X1_GDF3/VZ2.GDF3.R2"),
    # "DATA.STEMDIFF/X2_TIO2/VZ4.TIO2-A.M2.R2",
    # "DATA.STEMDIFF/X2_TIO2/VZ4.TIO2-R.M2.R2"
}
def filter_sd(d_path):
    d_path = Path(d_path)
    d = h5py.File(d_path, 'r')
    result = {}
    result_db = {}
    for name in d.keys():
        current_dataset = d[name]

        
        if len(list(paths[name].glob("*.dat"))) > 0:
            pattern = "*.dat"
        else:
            pattern = "??/*.dat"
        SDATA = sd.gvars.SourceData(
            detector  = sd.detectors.TimePix(),
            data_dir  = paths[name],
            filenames = pattern
        )

        db = sd.dbase.read_database(d_path.parent / "dbase" / f"db_{d_path.stem}_{name}")
        selected_idx = []
        for i in range(current_dataset.shape[0]):
            datafile = db.iloc[i]
            if datafile["Peaks"] > 1:
                selected_idx.append(i)
        
        result[name] = current_dataset[selected_idx]
        result_db[name] = db[db["Peaks"] > 1]

        # Verify dbase matches dataset
        for i in range(result[name].shape[0]):
            datafile = result_db[name].iloc[i]
            array_from_db = sd.io.Datafiles.read(SDATA, datafile["DatafileName"])
            assert (result[name][i] == array_from_db).all()

        print(result[name].shape)
        print(len(result_db[name]))
        
    return result, result_db


## Train

In [ ]:
result_train, result_train_db = filter_sd("dataset/train.h5")

(622, 256, 256)
622
(1447, 256, 256)
1447
(5215, 256, 256)
5215
(3757, 256, 256)
3757
(2757, 256, 256)
2757


In [4]:
save_h5(result_train, result_dir / "train.h5", "lzf")

In [5]:
for n, db in result_train_db.items():
    sd.dbase.save_database(db, result_dir / "dbase" / f"db_train_{n}")

## Val

In [ ]:
result_val, result_val_db = filter_sd("dataset/val.h5")

(134, 256, 256)
134
(309, 256, 256)
309
(1127, 256, 256)
1127
(803, 256, 256)
803
(598, 256, 256)
598


In [7]:
save_h5(result_val, result_dir / "val.h5")

In [8]:
for n, db in result_val_db.items():
    sd.dbase.save_database(db, result_dir / "dbase" / f"db_val_{n}")